# Latent Vandalism: The Joy of Productive Damage to Text-to-Image Synthesis Pipelines and Ornothology

**Workshop by Laura Wagner**

🔗 [laurajul.github.io](https://laurajul.github.io/)  
📦 [Workshop Repository](https://github.com/laurajul/latent-vandalism-workshop)

---

## Abstract

Text-to-image models have evolved into sophisticated engines of template culture (Grund and Scherffig), systems trained to reproduce standardized aesthetics. Fatigued by the constant flood of polished results and the arms race for images benchmarked on visual coherence, commercial value and consumer-friendliness, this workshop explores once again the charm of AI weirdness (Shane) - the failure in generative AI and the epistemic value of productive damage.

Drawing inspiration from glitch studies (Menkman), we embrace glitches and artifacts as revelatory moments. Through gently violating the consumer-friendly, polished norms meant to please, we surface the model's implicit assumptions about how things are supposed to look. Participants will work directly with Diffusion Transformers (DiT), focusing on the role of **embeddings** in image-text correlation from embedding space to latent space back into pixel space. Through hands-on meddling with the pipeline, we'll systematically **damage** and **reconfigure** the **semantic substrate** that guides image generation, deliberately perturbing inputs to understand this system's sensitivity and dynamics.

This counterfactual, gently adversarial approach, positions productive damage as a research method. Through **iatrogenic techniques** performed on text-to-image models, we probe the layers of technological inscription (Latour) embedded in these systems. Values, design choices, and visual norms inscribed become legible where the system breaks down. By deliberately coaxing the model into failure, we trace the contours of what has been encoded into them.

---

## High-level pipeline: text → embeddings → latent → pixels

**Text Prompt → Text Encoder → Embedding Space → Latent Space (diffusion) → VAE Decoder → Pixel Space**

1. **Embedding Space** (High-dimensional semantic vectors)
   - Where text meaning is encoded numerically from tokens

2. **Latent Space** (Compressed image representation)
   - Where diffusion actually happens
   - Much smaller than pixel space (e.g., 64×64×16 instead of 1024×1024×3)
   - Embeddings guide the denoising process here

3. **Pixel Space** (Final RGB image)
   - The inference result
   - Decoded from latent space by VAE

---

### Workshop Focus:

**We intervene at the Embedding Space** — modifying the text encoder's embeddings before they reach
the diffusion transformer, then let diffusion and the VAE decoder run normally from there.


## Summary

### Embedding Dimensions:
- **FLUX.2 [klein]-4B's text encoder (Qwen3-4B)**: 512 tokens × 7680 dimensions — three intermediate
  hidden layers (9, 18, 27 of 36) stacked side by side per token (3 × 2560), not just the final layer

### How It's Used:
- **Text embeddings** → Cross-attention in transformer (no separate pooled/global conditioning
  vector needed — unlike FLUX.1's T5 + CLIP pair, klein's single encoder's sequence embeddings do
  the whole job)

### Workshop Method:
- **Productive damage** as epistemological tool
- **Iatrogenic techniques** to probe system boundaries
- **Glitch aesthetics** as revelatory moments
- **Counterfactual experiments** to understand inscription

### What Vandalism Reveals:
- Direct manipulation bypasses text encoding limitations
- Systematic damage exposes training data biases
- Failures make visible the inscribed norms and assumptions
- AI weirdness provides epistemic value beyond polish
- Stacking early/mid/late hidden layers means damage at one token position touches multiple levels
  of linguistic abstraction at once, not just the encoder's final summary
- Template culture's boundaries become legible where it breaks


## Theoretical Framework: References

This workshop draws on several theoretical traditions:

### Glitch Studies
- **Menkman, Rosa.** *The Glitch Moment(um)*. Network Notebooks, 2011.
  - Glitches as revelatory moments that expose normally invisible structures
  - Productive failures as aesthetic and epistemic resources

### Template Culture
- **Grund, Katja and Scherffig, Lasse.** Work on template culture and standardized aesthetics in generative AI
  - How models reproduce homogeneous visual languages
  - The political economy of aesthetic standardization

### AI Weirdness
- **Shane, Janelle.** Research on AI failures and unexpected behaviors
  - The epistemic value of AI mistakes
  - How failures reveal system structure

### Science and Technology Studies
- **Latour, Bruno.** "Technology is society made durable." *Sociological Review*, 1990.
  - Technological inscription: How values and choices become embedded in systems
  - Making visible the social and political dimensions of technical artifacts

### Iatrogenic Methods
- Medical concept of harm caused by treatment itself, repurposed as deliberate intervention
  - Systematic damage as research methodology
  - Counterfactual reasoning through controlled failures

---

### About This Workshop

**Workshop by Laura Wagner**

🔗 Website: [laurajul.github.io](https://laurajul.github.io/)  
📦 Repository: [github.com/laurajul/latent-vandalism-workshop](https://github.com/laurajul/latent-vandalism-workshop)

For questions, feedback, or collaborations on productive damage to generative AI systems, please reach out via the website or repository.

---

*"The charm of AI weirdness is not just in the strange outputs, but in what those outputs reveal about the system that produced them."*

## 🎨 Running this workshop on Google Colab

This edition runs on a single free Colab **T4 GPU**, using **FLUX.2 [klein]-4B** instead of the
FLUX.1-schnell + T5-XXL + CLIP-L stack the university-cluster edition uses (still in the original
`notebooks/` for SD3.5 Medium too, where a shared filesystem makes several full model families
cheap). klein-4B uses a **single** text encoder (Qwen3-4B, no CLIP) and its whole quantized pipeline
is small enough (~5GB) that it's loaded **once, in full, at the start** — no load → unload → load
dance between sections. All five workshop steps — generate embeddings, save them to a library,
vandalize them, run FLUX inference, and the scaling animation — live in **this one notebook**, run
top to bottom.


## ⚙️ Colab Setup

Run the four cells below once at the start of the session. They:

1. Clone the workshop repo (for prompt files / configs — not model weights)
2. Mount your Google Drive and link the **shared, read-only** model folder (ask your workshop organizer for the link)
3. Install the extra packages Colab doesn't ship with
4. Set up your own private workspace folder plus the shared model path

**Why this looks different from the cluster version:** the original setup ran on a shared A100
cluster in `bfloat16`, downloading FLUX-Schnell *and* SD3.5 Medium (each with their own T5-XXL /
CLIP-L text encoders) into a shared model folder. This edition instead uses **FLUX.2 [klein]-4B**,
which needs none of that:

- **One model, one text encoder.** klein-4B pairs a 4B-parameter Qwen3 text encoder with a
  4B-parameter diffusion transformer — no CLIP, no second embedding to track.
- Loads **4-bit quantized (NF4)**, same as the FLUX.1 edition's T5/transformer did, but now that's
  the *entire* pipeline (~5GB total), not three separate multi-GB pieces.
- **Fully open (Apache 2.0), not gated** — no Hugging Face license click-through, no token required.
- Because the whole quantized pipeline is small, it's loaded **once** in the "Load Everything Once"
  section below and stays resident for the rest of the notebook — every later section (embedding
  generation, manipulation, FLUX inference, the scaling animation) reuses that same loaded pipeline
  instead of reloading anything.


In [1]:
# @title 1. Clone the workshop repository
from pathlib import Path

REPO_DIR = Path('/content/latent-vandalism-workshop')
if not REPO_DIR.exists():
    !git clone -q https://github.com/laurajul/latent-vandalism-workshop.git {REPO_DIR}
    print("✓ Cloned workshop repo")
else:
    print("✓ Repo already present")


✓ Cloned workshop repo


In [2]:
# @title 2. Mount Google Drive and link the shared models folder
from google.colab import drive
drive.mount('/content/drive')

from google.colab import auth
from googleapiclient.discovery import build
import re

# Define SHORTCUT_NAME globally here for all dependent cells
SHORTCUT_NAME = 'latent_vandalism_models'

drive_service = None
try:
    # Attempt to build drive_service without re-authenticating.
    # This might work if authentication from a previous run is still valid.
    drive_service = build('drive', 'v3')
    # Try a simple API call to ensure it's functional
    # This specific API call just lists one file, ensuring the service is active.
    drive_service.files().list(q="trashed=false", spaces='drive', fields='files(id, name)', pageSize=1).execute()
except Exception as e:
    # If building or testing drive_service fails, then re-authenticate.
    print(f"Initial Drive service check failed ({type(e).__name__}: {e}). Attempting authentication...")
    auth.authenticate_user() # This is the line that sometimes causes issues on re-run
    drive_service = build('drive', 'v3') # Build again after authentication

# ============================================================
# WORKSHOP ORGANIZER: paste the shared models folder link here.
# (created by notebooks_colab/00_instructor_model_prep.ipynb)
# and it will show up as this shortcut in every participant's Drive.
# Keep the real link out of git — paste it in locally and don't commit it.
# ============================================================
# SHORTCUT_NAME is now defined in cell 5982ba8d
SHARED_FOLDER_LINK = 'https://drive.google.com/drive/folders/PASTE_YOUR_FOLDER_ID_HERE?usp=drive_link'
# ============================================================

match = re.search(r'/folders/([a-zA-Z0-9_-]+)', SHARED_FOLDER_LINK)
if not match:
    raise ValueError('Could not extract a folder ID from SHARED_FOLDER_LINK — ask your organizer for the link')
folder_id = match.group(1)

existing = drive_service.files().list(
    q=f"name='{SHORTCUT_NAME}' and mimeType='application/vnd.google-apps.shortcut' and trashed=false",
    spaces='drive',
    fields='files(id, name)'
).execute().get('files', [])

if existing:
    print(f'✓ Shortcut "{SHORTCUT_NAME}" already exists in My Drive')
else:
    shortcut_metadata = {
        'name': SHORTCUT_NAME,
        'mimeType': 'application/vnd.google-apps.shortcut',
        'shortcutDetails': {'targetId': folder_id}
    }
    shortcut = drive_service.files().create(
        body=shortcut_metadata, fields='id, name, shortcutDetails'
    ).execute()
    print(f'✓ Created shortcut "{SHORTCUT_NAME}" in My Drive')

print(f'📁 Shared models available at: /content/drive/MyDrive/{SHORTCUT_NAME}/')


Mounted at /content/drive


Initial Drive service check failed (RefreshError: ("Failed to retrieve http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/?recursive=true from the Google Compute Engine metadata service. Status: 404 Response:\nb''", <google_auth_httplib2._Response object at 0x7b81bad016a0>)). Attempting authentication...


✓ Shortcut "latent_vandalism_models" already exists in My Drive
📁 Shared models available at: /content/drive/MyDrive/latent_vandalism_models/


In [3]:
# @title 3. Install packages
# torch + CUDA are pre-installed in Colab. Everything below is workshop-specific.
#
# diffusers>=0.40.0 (for Flux2KleinPipeline) requires huggingface-hub>=1.23.0,<2.0 — which every
# transformers 4.x release is incompatible with (they all cap huggingface-hub at <1.0). So
# transformers is pinned into the 5.x line here on purpose, matching
# 00_instructor_model_prep_local.ipynb's install cell — see its comment for the full story, including
# the one thing that costs us (a save_pretrained() quirk on 5.x that only matters for the *prep*
# notebook, since this notebook only ever loads the already-quantized checkpoint, never saves one).
# No -U here: Colab's torch/torchvision are preinstalled and CUDA-matched to the runtime's
# driver -- "-U" lets pip decide it also needs to upgrade torch to satisfy some package's
# minimum version, which can install a second, mismatched torch build and break CUDA import
# entirely (surfaces as a RuntimeError about overriding a dispatch key the moment you
# `import torch` next). Without -U, pip still installs whatever exact/ranged versions are
# requested below (nothing here is preinstalled on a fresh Colab runtime) but won't reach
# for a newer torch than what's already there.
%pip install -q "diffusers==0.40.0" "transformers==5.16.1" "huggingface_hub>=1.23.0,<2.0" "accelerate==1.12.0" bitsandbytes peft ipywidgets
# If you still hit "Trying to override a python impl for ... DispatchKey" on the next cell's
# `import torch`: torch already got touched in this VM. A kernel restart alone won't undo
# it -- use Runtime > Disconnect and delete runtime to get a genuinely fresh VM, then rerun
# from the top.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 112.4 MB/s eta 0:00:00


In [4]:
# @title Default title text
# transformers probes for TensorFlow and Flax/JAX backends on import if they're installed in the
# environment, even though this workshop only ever uses PyTorch — that's what the "Flax classes are
# deprecated" warnings you may see are from. Loading those unused backends costs real RAM on a
# RAM-constrained runtime like Colab's free T4, so turn them off before transformers/diffusers get
# imported anywhere in this notebook.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
import torch, json, shutil, gc
from pathlib import Path
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
else:
    print("  ⚠️ No GPU detected — go to Runtime > Change runtime type > T4 GPU")


# --- Shared, read-only: model weights, populated by the workshop organizer ---
MODELS_DIR = Path('/content/drive/MyDrive') / SHORTCUT_NAME
FLUX_MODEL_PATH = MODELS_DIR / "FLUX.2-klein-4B-nf4"
# Instructor provided embeddings (e.g., from a batch run)
INSTRUCTOR_EMBEDDINGS_PATH = MODELS_DIR / "example_embeddings.npz"

# --- Your own workspace: everyone gets their own folder, just like on the cluster ---
PROJECT_ROOT = Path('/content/drive/MyDrive/latent_vandalism_workshop')
DATA_DIR = PROJECT_ROOT / "data"
EMBEDDINGS_DIR = DATA_DIR / "embeddings"
OUTPUT_IMAGES_DIR = PROJECT_ROOT / "output/images"
SEQUENCE_DIR = DATA_DIR / "sequence"
# Participant's saved embeddings will be backed up here
LIBRARY_BACKUP_PATH = EMBEDDINGS_DIR / "embedding_library.npz"

for d in [EMBEDDINGS_DIR, OUTPUT_IMAGES_DIR, SEQUENCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Initialize global embedding library and dropdown list here
embedding_library = {}       # name -> {"prompt": str, "embedding": np.ndarray}
embedding_dropdowns = []     # every dropdown below that lists library entries, kept in sync

# Helper functions for managing the embedding library
def _load_embeddings_from_npz(filepath):
    """Helper to load embeddings from an .npz file into a dictionary."""
    if not filepath.exists():
        return {}
    data = np.load(filepath, allow_pickle=False)
    loaded_embeddings = {}
    for key in data.files:
        name, field = key.rsplit("::", 1)
        loaded_embeddings.setdefault(name, {})[field] = data[key]
    for name, entry in loaded_embeddings.items():
        entry["prompt"] = str(entry["prompt"])
    return loaded_embeddings

def save_library_backup():
    """Saves the current embedding_library (participant's additions) to their Drive."""
    # Only save embeddings that originated from the user, or modified ones.
    # For simplicity, this will save the *entire* current `embedding_library`,
    # potentially re-saving instructor embeddings if they were loaded.
    # A more complex system might track origin, but for this workshop, full save is fine.
    if not embedding_library:
        print("Library is empty — nothing to back up.")
        return
    payload = {}
    for name, entry in embedding_library.items():
        payload[f"{name}::embedding"] = entry["embedding"]
        payload[f"{name}::prompt"] = np.array(entry["prompt"])
    np.savez_compressed(LIBRARY_BACKUP_PATH, **payload)
    print(f"✓ Backed up {len(embedding_library)} embedding(s) to {LIBRARY_BACKUP_PATH}")
    print(f"  Size: {LIBRARY_BACKUP_PATH.stat().st_size / 1024:.1f} KB")

def refresh_embedding_dropdowns():
    """Updates the options of all registered embedding dropdown widgets."""
    names = list(embedding_library.keys())
    for dd in embedding_dropdowns:
        old_value = dd.value
        dd.options = names
        if old_value in names:
            dd.value = old_value
        elif names: # If old value is not in new options, try to set to first option
            dd.value = names[0]
        else: # No options available
            dd.value = None

# --- Initial Loading of Embeddings ---
# Load instructor embeddings first (read-only examples)
if INSTRUCTOR_EMBEDDINGS_PATH.exists():
    instructor_library = _load_embeddings_from_npz(INSTRUCTOR_EMBEDDINGS_PATH)
    embedding_library.update(instructor_library)
    print(f"✓ Loaded {len(instructor_library)} instructor embedding(s) from {INSTRUCTOR_EMBEDDINGS_PATH}")
else:
    print(f"ℹ️ No instructor embeddings found at {INSTRUCTOR_EMBEDDINGS_PATH}. This is expected if the instructor hasn't provided any.")

# Load participant's saved embeddings (from their private drive)
# These will overwrite instructor embeddings if names conflict (participant's data takes precedence).
if LIBRARY_BACKUP_PATH.exists():
    user_library = _load_embeddings_from_npz(LIBRARY_BACKUP_PATH)
    embedding_library.update(user_library)
    print(f"✓ Loaded {len(user_library)} participant embedding(s) from {LIBRARY_BACKUP_PATH}")
else:
    print(f"ℹ️ No participant embeddings found at {LIBRARY_BACKUP_PATH}. New embeddings will be saved here.")

refresh_embedding_dropdowns() # Update all dropdowns after initial loading

print(f"Total embeddings in library: {len(embedding_library)}")

Using device: cuda
  GPU: Tesla T4
ℹ️ No instructor embeddings found at /content/drive/MyDrive/latent_vandalism_models/example_embeddings.npz. This is expected if the instructor hasn't provided any.
✓ Loaded 16 participant embedding(s) from /content/drive/MyDrive/latent_vandalism_workshop/data/embeddings/embedding_library.npz
Total embeddings in library: 16


---

# FLUX.2 [klein]-4B: Load Everything Once 🦚🦜🐦
The Text-to-image model 🧊

Klein-4B's
quantized text encoder + transformer + VAE together are only ~5GB, comfortably resident on a free
T4 all at once. Run the cell below once and it will stay loaded for every section that follows.


In [5]:
# @title Load FLUX.2 [klein]-4B (text encoder + transformer + VAE — one pipeline, ~5GB total)
# ============================================================
# All three real components are loaded straight onto the GPU and stay there for the rest of the
# notebook — no `enable_model_cpu_offload()` needed, no unloading between sections. That's the whole
# point of switching to klein-4B on a free T4: FLUX.1-schnell's transformer alone (~6.8GB NF4) plus
# T5-XXL (~6GB NF4) plus CLIP-L was already close to the T4's 16GB VRAM ceiling and Colab's system RAM
# ceiling, forcing the old edition to load/unload each one in turn. klein-4B's whole pipeline fits in
# roughly a third of that.
# ============================================================
import torch
from diffusers import Flux2KleinPipeline, Flux2Transformer2DModel, AutoencoderKLFlux2
from diffusers import FlowMatchEulerDiscreteScheduler
from transformers import Qwen3ForCausalLM, Qwen2TokenizerFast

# SHORTCUT_NAME is now defined in cell 5982ba8d

# Ensure bitsandbytes is updated to meet diffusers requirements
%pip install -U bitsandbytes

if not FLUX_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"No pre-quantized FLUX.2 [klein]-4B checkpoint at {FLUX_MODEL_PATH}. This cell only loads an "
        "already-prepared model from the shared Drive folder — it doesn't download/quantize anything "
        "itself (that's 00_instructor_model_prep_local.ipynb's job, run once by the workshop "
        "organizer). Check that Drive is mounted and the shared folder is linked under the right name."
    )

print("Loading FLUX.2 [klein]-4B from the shared Drive folder...")
# Joined path passed directly (no subfolder= kwarg): subfolder-joining for a *local* directory has
# been unreliable for some diffusers model classes as of this writing (surfaces as "Error no file
# named config.json found in directory ..." even though the file is right there in the subfolder) —
# passing the already-joined path sidesteps it entirely.
tokenizer = Qwen2TokenizerFast.from_pretrained(FLUX_MODEL_PATH / "tokenizer", local_files_only=True)
text_encoder = Qwen3ForCausalLM.from_pretrained(
    FLUX_MODEL_PATH / "text_encoder", local_files_only=True, device_map="auto",
)
transformer = Flux2Transformer2DModel.from_pretrained(
    FLUX_MODEL_PATH / "transformer", torch_dtype=torch.float16, local_files_only=True,
    device_map="auto", low_cpu_mem_usage=True,
)
vae = AutoencoderKLFlux2.from_pretrained(
    FLUX_MODEL_PATH / "vae", torch_dtype=torch.float16, local_files_only=True,
).to(device)
scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(FLUX_MODEL_PATH / "scheduler", local_files_only=True)

flux_pipe = Flux2KleinPipeline(
    scheduler=scheduler, vae=vae,
    text_encoder=text_encoder, tokenizer=tokenizer,
    transformer=transformer,
    is_distilled=True,  # klein is a distilled model, like FLUX.1-schnell — this skips real
                         # classifier-free guidance at generation time (see the Inference section)
)

print("✓ FLUX.2 [klein]-4B loaded — text encoder, transformer, and VAE all stay resident for the")
print("  rest of this notebook. No unload/reload needed between the sections below.")


Loading FLUX.2 [klein]-4B from the shared Drive folder...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/debugpy/_vendored/pydevd/_pydevd_bundle/pydevd_safe_repr.py:128: FutureWarning: Accessing config attribute `__iter__` directly via 'Flux2Transformer2DModel' object attribute is deprecated. Please access '__iter__' over 'Flux2Transformer2DModel's config object instead, e.g. 'unet.config.__iter__'.
  if not hasattr(obj, "__iter__"):


✓ FLUX.2 [klein]-4B loaded — text encoder, transformer, and VAE all stay resident for the
  rest of this notebook. No unload/reload needed between the sections below.


---

# Text Embeddings with FLUX.2 [klein]'s Qwen3 Encoder 🦚🦜🐦

## Generate Single Embedding

Enter a prompt below and click **Generate Embedding**. This runs the same tokenize → chat-template →
forward-pass → pick-three-hidden-layers → reshape recipe `Flux2KleinPipeline` uses internally, exposed
here as a plain array so you can look at it and manipulate it directly instead of treating the encoder
as a black box. Every embedding you generate is saved to your library automatically, under its prompt
text — see the next section if you'd rather save it again under a custom name.


In [6]:
# @title Generate Single Embedding


MAX_SEQUENCE_LENGTH = 512
HIDDEN_STATE_LAYERS = (9, 18, 27)  # Flux2KleinPipeline's own defaults, out of Qwen3-4B's 36 layers

def encode_prompt_klein(prompt):
    """Reproduces Flux2KleinPipeline._get_qwen3_prompt_embeds by hand, returning a plain numpy array
    instead of a tensor buried inside the pipeline."""
    messages = [{"role": "user", "content": prompt}]
    chat_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    tokens = tokenizer(
        chat_text, return_tensors="pt", padding="max_length", truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
    )
    token_strings = tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])
    num_real_tokens = int(tokens["attention_mask"][0].sum())

    with torch.no_grad():
        tokens = {k: v.to(device) for k, v in tokens.items()}
        output = text_encoder(
            input_ids=tokens["input_ids"],
            attention_mask=tokens["attention_mask"],
            output_hidden_states=True, use_cache=False,
        )
        # Stack 3 intermediate layers instead of using just the last one, then fold them into one
        # wider per-token vector: [1, 3, 512, 2560] -> [1, 512, 3*2560]
        stacked = torch.stack([output.hidden_states[k] for k in HIDDEN_STATE_LAYERS], dim=1)
        _, num_layers, seq_len, hidden_dim = stacked.shape
        embedding = stacked.permute(0, 2, 1, 3).reshape(1, seq_len, num_layers * hidden_dim)

    return embedding.float().cpu().numpy()[0], token_strings, num_real_tokens


# Create text input widget
prompt_input = widgets.Textarea(
    value='a puffy european robin sitting on a tree branch',
    placeholder='Enter your prompt here',
    description='Prompt:',
    layout=widgets.Layout(width='80%', height='80px')
)
generate_button = widgets.Button(description='Generate Embedding', button_style='success')
output_area = widgets.Output()

current_embedding = None
current_prompt = None

def generate_embedding(b):
    global current_embedding, current_prompt
    with output_area:
        output_area.clear_output()
        prompt = prompt_input.value
        print(f"Generating embedding for: '{prompt}'\n")

        current_embedding, token_strings, num_real_tokens = encode_prompt_klein(prompt)
        current_prompt = prompt

        print(f"Tokenized into {num_real_tokens} real tokens (+ {MAX_SEQUENCE_LENGTH - num_real_tokens} padding):")
        print("First 10 tokens:", token_strings[:10])
        print()
        print(f"✓ Embedding generated!")
        print(f"  Shape: {current_embedding.shape}  (512 tokens × 3 stacked hidden layers × 2560 dims)")
        print(f"  Size: {current_embedding.nbytes / 1024:.2f} KB")
        print()
        print(f"First token embedding (first 10 values):")
        print(current_embedding[0, :10])

        # Saved to the library by default — no separate "Save to Library" click needed. That
        # section further down is only for saving this same embedding again under a custom name.
        default_name = current_prompt[:60]
        if default_name in embedding_library:
            print(f"\n⚠️ Overwriting existing library entry '{default_name}'.")
        embedding_library[default_name] = {"prompt": current_prompt, "embedding": current_embedding.copy()}
        refresh_embedding_dropdowns()
        save_library_backup()
        print(f"\n✓ Saved to library as '{default_name}' — library now has {len(embedding_library)} embedding(s).")

generate_button.on_click(generate_embedding)
display(prompt_input, generate_button, output_area)


Textarea(value='a puffy european robin sitting on a tree branch', description='Prompt:', layout=Layout(height=…

Button(button_style='success', description='Generate Embedding', style=ButtonStyle())

Output()

## Save to the Embedding Library

Generating an embedding above already saved it to your library automatically, under its prompt text,
and backed it up to your Drive — no extra step required. Use the section below only if you want to
save that same embedding again under a different, custom name (e.g. something shorter and easier to
spot later in the dropdowns).

No JSON round-trip: saved embeddings live as plain variables in this notebook's memory, in a small
dict (`embedding_library`), and every dropdown further down (Manipulation, Image Generation, Scaling
Animation) reads from that same dict.


In [7]:
name_input = widgets.Text(
    value='', placeholder='optional — defaults to the prompt text',
    description='Save as:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%'),
)
save_button = widgets.Button(description='Save to Library', button_style='primary')
save_output = widgets.Output()

def save_embedding(b):
    with save_output:
        save_output.clear_output()
        if current_embedding is None:
            print("❌ No embedding to save! Generate one above first.")
            return
        name = name_input.value.strip() or current_prompt[:60]
        if name in embedding_library:
            print(f"⚠️ Overwriting existing embedding '{name}'.")
        embedding_library[name] = {"prompt": current_prompt, "embedding": current_embedding.copy()}
        refresh_embedding_dropdowns()
        save_library_backup() # Persist to user's local drive
        print(f"✓ Saved as '{name}' — library now has {len(embedding_library)} embedding(s):")
        for n in embedding_library:
            print(f"  - {n}")

save_button.on_click(save_embedding)
display(name_input, save_button, save_output)

Text(value='', description='Save as:', layout=Layout(width='60%'), placeholder='optional — defaults to the pro…

Button(button_style='primary', description='Save to Library', style=ButtonStyle())

Output()

---

# Embedding Manipulation

Pick one saved embedding ("A") and manipulate it directly — scale it, invert it, zero out a range of
token positions, or combine it with a second saved embedding ("B"): average, add, or subtract.
Every operation here works on plain numpy arrays; the heatmap below is the only "conversion" this
section does — no file format involved.


In [9]:
import matplotlib.pyplot as plt

manip_current_name = None
manip_current_embedding = None
manip_current_prompt = None
manip_secondary_name = None
manip_modified_embedding = None
manip_zero_range = None

print(f"✓ Setup complete! {len(embedding_library)} embedding(s) currently in the library.")


✓ Setup complete! 16 embedding(s) currently in the library.


In [10]:
# @title Embedding Manipulation
#blabal
import matplotlib.pyplot as plt
import ipywidgets as widgets # Explicitly import widgets
import numpy as np # Explicitly import numpy

# Defensive initialization for global variables that should have been set up
if 'embedding_library' not in globals():
    embedding_library = {}
if 'embedding_dropdowns' not in globals():
    embedding_dropdowns = []

# These variables should be managed by the functions, not re-initialized here
# manip_current_name = None
# manip_current_embedding = None
# manip_current_prompt = None
# manip_secondary_name = None
# manip_modified_embedding = None
# manip_zero_range = None

# Removed redundant setup complete print from here, it's in the previous cell

# NEW: Output widget for matplotlib plots
visualization_output = widgets.Output()

# ==================== LOAD SECTION ====================
embedding_a_dropdown = widgets.Dropdown(
    options=list(embedding_library.keys()), description='Embedding A:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'),
)
# Removed embedding_b_dropdown from this section as arithmetic operations are moved
embedding_dropdowns.append(embedding_a_dropdown) # Only append A dropdown for this section

load_button = widgets.Button(description='Load Embedding A', button_style='success')
load_output = widgets.Output()

def load_embedding(b):
    global manip_current_name, manip_current_embedding, manip_current_prompt
    global manip_modified_embedding, manip_zero_range # Ensure manip_modified_embedding is reset
    with load_output:
        load_output.clear_output()
        name = embedding_a_dropdown.value
        if not name:
            print("❌ No embedding selected! Save one in the Text Embeddings section first.")
            return
        entry = embedding_library[name]
        manip_current_name = name
        manip_current_embedding = entry["embedding"]
        manip_current_prompt = entry["prompt"]
        manip_modified_embedding = None # Reset modified embedding when a new original is loaded
        manip_zero_range = None # Reset zero range

        num_tokens = manip_current_embedding.shape[0]
        zero_range_slider.max = num_tokens
        zero_range_slider.value = (0, num_tokens)

        print(f"✓ Loaded '{name}'")
        print(f"  Prompt: '{manip_current_prompt}'")
        print(f"  Shape: {manip_current_embedding.shape}")
        print(f"  Value range: [{manip_current_embedding.min():.4f}, {manip_current_embedding.max():.4f}]")

        visualize_comparison() # Call visualization after loading

load_button.on_click(load_embedding)

# ==================== MANIPULATION SECTION ====================
manipulation_dropdown = widgets.Dropdown(
    options=['Scale', 'Invert', 'Zero Range'], # Removed arithmetic operations
    value='Scale', description='Operation:', style={'description_width': 'initial'},
)
scale_slider = widgets.FloatSlider(
    value=1.0, min=-3.65, max=3.65, step=0.01, description='Scale Factor:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'),
)
zero_range_slider = widgets.IntRangeSlider(
    value=(0, 512), min=0, max=512, step=1, description='Keep Range:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'),
    continuous_update=True,
)
zero_range_label = widgets.HTML(value="<b>Keeping all positions</b> (no zeroing)")
params_box = widgets.VBox([scale_slider])

def update_params(change):
    manipulation = change['new']
    if manipulation == 'Scale':
        params_box.children = [scale_slider]
    elif manipulation == 'Zero Range':
        params_box.children = [zero_range_slider, zero_range_label]
    else: # Invert and any other future single-parameter operations
        params_box.children = []

def update_range_label(change):
    start, end = change['new']
    max_val = zero_range_slider.max
    zeroed_parts = []
    if start > 0:
        zeroed_parts.append(f"positions 0-{start-1}")
    if end < max_val:
        zeroed_parts.append(f"positions {end}-{max_val-1}")
    zero_range_label.value = (
        f"<b>Zeroing:</b> {', '.join(zeroed_parts)} | <b>Keeping:</b> positions {start}-{end-1}"
        if zeroed_parts else "<b>Keeping all positions</b> (no zeroing)"
    )

apply_button = widgets.Button(description='Apply Manipulation', button_style='warning')
manipulation_output = widgets.Output()

def apply_manipulation(b):
    global manip_modified_embedding, manip_zero_range, manip_secondary_name
    with manipulation_output:
        manipulation_output.clear_output()
        if manip_current_embedding is None:
            print("❌ No embedding loaded! Load Embedding A first.")
            return

        manipulation = manipulation_dropdown.value
        modified = manip_current_embedding.copy()
        manip_zero_range = None

        # Removed arithmetic operations from this section

        if manipulation == 'Scale':
            factor = scale_slider.value
            modified = modified * factor
            print(f"✓ Scaled embedding by {factor}x")

        elif manipulation == 'Invert':
            modified = -modified
            print(f"✓ Inverted embedding values")

        elif manipulation == 'Zero Range':
            start, end = zero_range_slider.value
            manip_zero_range = (start, end)
            num_tokens = modified.shape[0]
            if start > 0:
                modified[:start] = 0.0
            if end < num_tokens:
                modified[end:] = 0.0
            print(f"✓ Zeroed token positions outside range [{start}, {end})")

        manip_modified_embedding = modified
        print(f"  Original range: [{manip_current_embedding.min():.4f}, {manip_current_embedding.max():.4f}]")
        print(f"  Modified range: [{modified.min():.4f}, {modified.max():.4f}]")
        print(f"  Shape: {manip_modified_embedding.shape}")

        # Call the visualization function here
        visualize_comparison()

apply_button.on_click(apply_manipulation)

# ==================== VISUALIZATION ====================
def add_zero_range_lines(ax, zero_range, num_tokens):
    if zero_range is not None:
        start, end = zero_range
        # Plot lines at the boundary of the "keep" range
        if start > 0:
            ax.axvline(x=start-0.5, color='lime', linewidth=2, linestyle='--', label='Keep range start')
        if end < num_tokens:
            ax.axvline(x=end-0.5, color='lime', linewidth=2, linestyle='--', label='Keep range end')
        if start > 0 or end < num_tokens: # Only show legend if lines are actually drawn
            # Prevent duplicate labels if both lines are drawn
            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if 'Keep range start' in by_label and 'Keep range end' in by_label:
                # If both are present, simplify the legend entry
                ax.legend([by_label['Keep range start']], ['Keep range'], loc='upper right')
            else:
                ax.legend(loc='upper right')

def visualize_comparison():
    # This function now outputs to the dedicated visualization_output widget
    with visualization_output:
        visualization_output.clear_output() # Clear previous plot and text

        if manip_current_embedding is None:
            print("❌ No embedding loaded yet. Load an embedding to visualize.")
            return

        original = manip_current_embedding
        num_tokens, num_dims = original.shape
        vmin, vmax = -0.3, 0.3
        truncate_dims = 512

        if manip_modified_embedding is not None:
            # Two plots: Original vs. Modified
            modified = manip_modified_embedding
            fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
            prompt_display = f'"{(manip_current_prompt[:40] + "...") if len(manip_current_prompt) > 40 else manip_current_prompt}"'
            manipulation_name = manipulation_dropdown.value

            panels = [
                (axes[0], original[:, :truncate_dims], f'Original: {prompt_display}\n(dims 0-{truncate_dims})'),
                (axes[1], modified[:, :truncate_dims], f'Modified ({manipulation_name})\n(dims 0-{truncate_dims})'),
            ]
            for ax, data, title in panels:
                im = ax.imshow(data.T, aspect='auto', cmap='RdBu_r', vmin=vmin, vmax=vmax)
                ax.set_title(title, fontweight='bold', fontsize=10)
                ax.set_xlabel('Token Position')
                ax.set_ylabel('Dimension')
                plt.colorbar(im, ax=ax, label='Value', shrink=0.8)
                add_zero_range_lines(ax, manip_zero_range, num_tokens)
            plt.show()

            # Print comparison statistics only if both are present
            print("\n" + "="*60)
            print("COMPARISON STATISTICS")
            print("="*60)
            print(f"\n{'Metric':<25} {'Original':<15} {'Modified':<15} {'Change':<15}")
            print("-"*70)
            print(f"{'Min value':<25} {original.min():.4f} {modified.min():.4f} {modified.min()-original.min():<+15.4f}")
            print(f"{'Max value':<25} {original.max():.4f} {modified.max():.4f} {modified.max()-original.max():<+15.4f}")
            print(f"{'Mean':<25} {original.mean():.4f} {modified.mean():.4f} {modified.mean()-original.mean():<+15.4f}")
            print(f"{'Std deviation':<25} {original.std():.4f} {modified.std():.4f} {modified.std()-original.std():<+15.4f}")
            print(f"{'Total L2 norm':<25} {np.linalg.norm(original):.4f} {np.linalg.norm(modified):.4f} {np.linalg.norm(modified)-np.linalg.norm(original):<+15.4f}")

            # Check for zero norm to prevent division by zero for cos_sim
            original_norm = np.linalg.norm(original)
            modified_norm = np.linalg.norm(modified)
            if original_norm > 1e-9 and modified_norm > 1e-9: # small epsilon to avoid near-zero
                cos_sim = np.dot(original.flatten(), modified.flatten()) / (original_norm * modified_norm)
                print(f"\n{'Cosine similarity':<25} {cos_sim:.6f}")
            else:
                print("\nCosine similarity not calculable due to zero or near-zero norm.")
            print("="*60)

        else:
            # One plot: only the original embedding (on initial load)
            fig, ax = plt.subplots(1, 1, figsize=(8, 7), constrained_layout=True) # Single subplot
            prompt_display = f'"{(manip_current_prompt[:40] + "...") if len(manip_current_prompt) > 40 else manip_current_prompt}"'
            im = ax.imshow(original[:, :truncate_dims].T, aspect='auto', cmap='RdBu_r', vmin=vmin, vmax=vmax)
            ax.set_title(f'Loaded: {prompt_display}\n(dims 0-{truncate_dims})', fontweight='bold', fontsize=10)
            ax.set_xlabel('Token Position')
            ax.set_ylabel('Dimension')
            plt.colorbar(im, ax=ax, label='Value', shrink=0.8)
            plt.show()

            print("\n" + "="*60)
            print("LOADED EMBEDDING STATISTICS")
            print("="*60)
            print(f"\n{'Metric':<25} {'Value':<15}")
            print("-"*40)
            print(f"{'Min value':<25} {original.min():.4f}")
            print(f"{'Max value':<25} {original.max():.4f}")
            print(f"{'Mean':<25} {original.mean():.4f}")
            print(f"{'Std deviation':<25} {original.std():.4f}")
            print(f"{'Total L2 norm':<25} {np.linalg.norm(original):.4f}")
            print("="*60)


# visualize_button.on_click(visualize_comparison) # Removed: now automatic

# ==================== SAVE SECTION ====================
save_name_input = widgets.Text(
    value='', placeholder='optional — auto-named from the operation', description='Save as:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='60%'),
)
save_modified_button = widgets.Button(description='Save Modified Embedding', button_style='primary')
save_modified_output = widgets.Output()

def save_modified(b):
    with save_modified_output:
        save_modified_output.clear_output()
        if manip_modified_embedding is None:
            print("❌ No modified embedding to save! Apply a manipulation first.")
            return

        manipulation = manipulation_dropdown.value
        # No more arithmetic operations in this section
        if manipulation == 'Scale':
            suffix = f"_scaled_{scale_slider.value}x"
        elif manipulation == 'Invert':
            suffix = "_inverted"
        elif manipulation == 'Zero Range':
            start, end = zero_range_slider.value
            suffix = f"_keep_{start}_to_{end}"
        else:
            suffix = "_modified" # Fallback for new operations

        default_name = f"{manip_current_name}{suffix}"
        name = save_name_input.value.strip() or default_name

        prompt = manip_current_prompt # Simplified prompt for single-embedding manipulations
        embedding_library[name] = {"prompt": prompt, "embedding": manip_modified_embedding.copy()}
        refresh_embedding_dropdowns()
        print(f"✓ Saved as '{name}' — library now has {len(embedding_library)} embedding(s)")

save_modified_button.on_click(save_modified)

manipulation_dropdown.observe(update_params, names='value')
zero_range_slider.observe(update_range_label, names='value')

# ==================== DISPLAY ====================
# Wrap the control widgets in a VBox
control_panel = widgets.VBox([
    widgets.HTML("<h3>1. Load Embedding(s)</h3>"),
    embedding_a_dropdown, # embedding_b_dropdown removed from this display
    load_button, load_output,
    widgets.HTML("<hr><h3>2. Apply Manipulation & Visualize Changes</h3>"),
    manipulation_dropdown, params_box, apply_button, manipulation_output,
    widgets.HTML("<hr><h3>3. Save Modified Embedding</h3>"),
    save_name_input, save_modified_button, save_modified_output,
])

# Display control panel and visualization output side-by-side
display(widgets.HBox([control_panel, visualization_output]))

---

# Embedding Arithmetic (A + B, A - B, (A + B) / 2)

In [11]:
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt

# Global variables for merge embeddings
# These will store the result of the merge operation
arith_result_embedding = None
arith_result_prompt = None
arith_result_name = None

# Output widgets for this section
arith_manipulation_output = widgets.Output()
arith_visualization_output = widgets.Output()
arith_save_output = widgets.Output()


# ==================== MULTI-EMBEDDING MERGE ====================
NUM_MERGE_SLOTS = 5
merge_embedding_dropdowns = []
merge_strength_inputs = []
merge_slots_hboxes = []

for i in range(NUM_MERGE_SLOTS):
    # Add an empty option to dropdowns so slots can be left unused
    dropdown = widgets.Dropdown(
        options=[''] + list(embedding_library.keys()),
        description=f'Embedding {i+1}:',
        style={'description_width': 'initial'}, layout=widgets.Layout(width='400px'),
    )
    strength_input = widgets.FloatSlider(
        value=1.0, min=0.0, max=1.0, step=0.01, description='Strength:',
        style={'description_width': 'initial'}, layout=widgets.Layout(width='200px'), # Adjust width for slider
    )
    # Add to the global embedding_dropdowns list so refresh_embedding_dropdowns updates it
    embedding_dropdowns.append(dropdown)
    merge_embedding_dropdowns.append(dropdown)
    merge_strength_inputs.append(strength_input)
    merge_slots_hboxes.append(widgets.HBox([dropdown, strength_input]))

apply_merge_button = widgets.Button(description='Apply Merge', button_style='warning')

def apply_merge(b):
    global arith_result_embedding, arith_result_prompt, arith_result_name
    with arith_manipulation_output:
        arith_manipulation_output.clear_output()

        active_embeddings = []
        active_prompts = []
        active_names = []
        active_strengths = []

        for i in range(NUM_MERGE_SLOTS):
            name = merge_embedding_dropdowns[i].value
            strength = merge_strength_inputs[i].value
            if name and strength != 0: # Only consider active slots with non-zero strength
                entry = embedding_library[name]
                active_embeddings.append(entry["embedding"])
                active_prompts.append(entry["prompt"])
                active_names.append(name)
                active_strengths.append(strength)

        if not active_embeddings:
            print("❌ No embeddings selected or all strengths are zero for merging.")
            return
        if len(active_embeddings) == 1:
            print("ℹ️ Only one embedding selected. Result will be a scaled version of itself.")

        # Calculate weighted sum
        weighted_sum_embedding = np.zeros_like(active_embeddings[0])
        for emb, strength in zip(active_embeddings, active_strengths):
            weighted_sum_embedding += emb * strength

        arith_result_embedding = weighted_sum_embedding

        # Generate result prompt and name
        prompt_parts = []
        name_parts = []
        for i, name in enumerate(active_names):
            # Truncate prompt part for display, keep name part shorter too
            prompt_parts.append(f"('{active_prompts[i][:20]}...' * {active_strengths[i]:.2f})")
            name_parts.append(f"{name[:20]}_{active_strengths[i]:.2f}")

        arith_result_prompt = " + ".join(prompt_parts)
        # Ensure the generated name is not too long for filesystem/display
        arith_result_name = "merged_" + "_".join(name_parts)
        if len(arith_result_name) > 100: # Arbitrary limit for readability
            arith_result_name = arith_result_name[:97] + "..."

        print(f"✓ Applied Multi-Embedding Merge (Total {len(active_embeddings)} embeddings)")
        print(f"  Resulting embedding shape: {arith_result_embedding.shape}")
        print(f"  Resulting embedding value range: [{arith_result_embedding.min():.4f}, {arith_result_embedding.max():.4f}]")
        visualize_merge_comparison(active_embeddings, active_prompts, active_names, active_strengths)

apply_merge_button.on_click(apply_merge)


# ==================== VISUALIZATION ====================
def visualize_merge_comparison(active_inputs, active_input_prompts, active_input_names, active_strengths):
    with arith_visualization_output:
        arith_visualization_output.clear_output()

        embeddings_to_plot = []
        titles = []

        # Visualize up to first 2 input embeddings for clarity, plus the result
        for i in range(min(len(active_inputs), 2)):
            embeddings_to_plot.append(active_inputs[i])
            titles.append(f"Input {i+1}: '{active_input_names[i]}' (x{active_strengths[i]:.2f})\n({active_input_prompts[i][:40]}...)")

        if len(active_inputs) > 2:
            # Add a note if more inputs were used but not visualized individually
            if len(titles) > 0:
                titles[-1] += f" (+{len(active_inputs)-len(embeddings_to_plot)} more inputs)"
            else: # If no inputs were visualized due to being too many, but result is there.
                pass # The result title will convey the merge.

        if arith_result_embedding is not None:
            embeddings_to_plot.append(arith_result_embedding)
            titles.append(f"Merged Result: '{arith_result_name}'\n({arith_result_prompt[:40]}...)")

        if not embeddings_to_plot:
            print("Select embeddings and apply merge to visualize them.")
            return

        num_plots = len(embeddings_to_plot)
        # Adjust figure size dynamically based on number of plots
        fig, axes = plt.subplots(1, num_plots, figsize=(min(6 * num_plots, 18), 7), constrained_layout=True)

        # Handle case of single plot (axes is not an array then)
        if num_plots == 1:
            axes = [axes]

        vmin, vmax = -0.3, 0.3 # Consistent color scale
        truncate_dims = 512 # Display only the first 512 dimensions

        for i, (ax, emb, title) in enumerate(zip(axes, embeddings_to_plot, titles)):
            im = ax.imshow(emb[:, :truncate_dims].T, aspect='auto', cmap='RdBu_r', vmin=vmin, vmax=vmax)
            ax.set_title(title, fontweight='bold', fontsize=10)
            ax.set_xlabel('Token Position')
            ax.set_ylabel('Dimension')
            if i == num_plots - 1: # Add colorbar only to the last plot for clarity
                plt.colorbar(im, ax=ax, label='Value', shrink=0.8)
        plt.show()


# ==================== SAVE RESULTING EMBEDDING ====================
save_arith_name_input = widgets.Text(
    value='', placeholder='optional — auto-named from operation', description='Save as:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='60%'),
)
save_arith_result_button = widgets.Button(description='Save Resulting Embedding', button_style='primary')

def save_arith_result(b):
    global arith_result_name
    with arith_save_output:
        arith_save_output.clear_output()
        if arith_result_embedding is None:
            print("❌ No resulting embedding to save! Apply an arithmetic operation first.")
            return

        default_name = arith_result_name if arith_result_name else "merged_embedding_result"
        name = save_arith_name_input.value.strip() or default_name

        if name in embedding_library:
            print(f"⚠️ Overwriting existing embedding '{name}'.")
        embedding_library[name] = {"prompt": arith_result_prompt, "embedding": arith_result_embedding.copy()}
        refresh_embedding_dropdowns()
        save_library_backup() # Persist to user's local drive
        print(f"✓ Saved as '{name}' — library now has {len(embedding_library)} embedding(s)")

save_arith_result_button.on_click(save_arith_result)


# ==================== DISPLAY ALL WIDGETS ====================
arith_control_panel = widgets.VBox([
    widgets.HTML("<h3>1. Select Embeddings & Strengths for Merge</h3>"),
    *merge_slots_hboxes, # Unpack the list of HBox widgets for each slot
    arith_manipulation_output, # For messages from apply_merge
    widgets.HTML("<hr>"),
    apply_merge_button,
    widgets.HTML("<hr><h3>2. Save Resulting Embedding</h3>"),
    save_arith_name_input, save_arith_result_button, arith_save_output,
])

# Display control panel and visualization output side-by-side
display(widgets.HBox([arith_control_panel, arith_visualization_output]))

---

# FLUX.2 [klein] Inference

## Generate Image from Embedding

`flux_pipe` was already loaded in full at the top of the notebook — this just feeds it a
(possibly-vandalized) embedding from the library instead of a raw prompt string.


---

In [13]:
import ipywidgets as widgets
from IPython.display import display

# The global embedding_library and embedding_dropdowns are still used.
# The refresh_embedding_dropdowns function is also assumed to be global.
# (These are defined in earlier cells, e.g., 0a227f77)

# The make_polaroid function is also assumed to be globally available (defined in 5a73d3d7)
# The flux_pipe, device, etc. are also assumed to be global (defined in 545643ce and 5a73d3d7)

def make_polaroid(image, label_lines):
    """Polaroid-style frame: rounded corners and a soft drop shadow around the card, the
    generated image on top, a caption below. Everything outside the card and its shadow is
    transparent (RGBA), rather than sitting on a white/gray canvas."""
    from PIL import ImageDraw, ImageFont, ImageFilter

    image = image.convert('RGB')
    orig_width, orig_height = image.size
    border = 20
    corner_radius = 8
    shadow_blur, shadow_offset, shadow_margin = 14, 5, 26

    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 32)
    except Exception:
        font = ImageFont.load_default()

    card_width = orig_width + 2 * border

    # A caption longer than the card is wide gets cut off with an ellipsis instead of
    # overflowing the card or wrapping onto another line.
    max_text_width = card_width - 2 * border
    def fit_line(text):
        if font.getlength(text) <= max_text_width:
            return text
        ellipsis = "..."
        while text and font.getlength(text + ellipsis) > max_text_width:
            text = text[:-1]
        return (text + ellipsis) if text else ellipsis

    label_lines = [fit_line(line) for line in label_lines]

    line_gap = 8
    line_height = font.getbbox("Ag")[3] - font.getbbox("Ag")[1]
    caption_height = 20 + len(label_lines) * (line_height + line_gap)
    card_height = orig_height + border + caption_height

    # Build the card itself (photo + caption) as a normal RGBA image first.
    card = Image.new('RGBA', (card_width, card_height), (255, 255, 255, 255))
    card.paste(image, (border, border))
    draw = ImageDraw.Draw(card)
    y = orig_height + border + 10
    for line in label_lines:
        bbox = draw.textbbox((0, 0), line, font=font)
        x = (card_width - (bbox[2] - bbox[0])) // 2
        draw.text((x, y), line, fill='#222222', font=font)
        y += (bbox[3] - bbox[1]) + line_gap

    # Round the card's corners via an alpha mask.
    corner_mask = Image.new('L', (card_width, card_height), 0)
    ImageDraw.Draw(corner_mask).rounded_rectangle(
        [0, 0, card_width - 1, card_height - 1], radius=corner_radius, fill=255,
    )
    card.putalpha(corner_mask)

    # Compose onto a slightly larger *transparent* canvas so there's room for a soft, blurred
    # shadow — drawn faint (low alpha) and offset a little below the card, then blurred. Nothing
    # fills the canvas itself, so only the card and its shadow are opaque; everywhere else stays
    # transparent.
    canvas_width, canvas_height = card_width + 2 * shadow_margin, card_height + 2 * shadow_margin
    shadow_mask = Image.new('L', (canvas_width, canvas_height), 0)
    ImageDraw.Draw(shadow_mask).rounded_rectangle(
        [shadow_margin, shadow_margin + shadow_offset,
         shadow_margin + card_width - 1, shadow_margin + shadow_offset + card_height - 1],
        radius=corner_radius, fill=45,  # low alpha — a very light shadow
    )
    shadow_mask = shadow_mask.filter(ImageFilter.GaussianBlur(shadow_blur))
    shadow_layer = Image.new('RGBA', (canvas_width, canvas_height), (0, 0, 0, 0))
    shadow_layer.putalpha(shadow_mask)

    canvas = Image.new('RGBA', (canvas_width, canvas_height), (0, 0, 0, 0))
    canvas.alpha_composite(shadow_layer)
    canvas.alpha_composite(card, (shadow_margin, shadow_margin))

    return canvas


def create_image_generation_vbox():
    # Variables to hold the last generated image and its name, accessible by the 'send to gallery' button
    last_generated_polaroid = None
    last_generated_name = None

    # Instantiate all widgets for this specific VBox
    local_gen_embedding_dropdown = widgets.Dropdown(
        options=list(embedding_library.keys()), description='Embedding:',
        style={'description_width': 'initial'}, layout=widgets.Layout(width='600px'),
    )
    # Add this local dropdown to the global list so refresh_embedding_dropdowns can update it
    embedding_dropdowns.append(local_gen_embedding_dropdown)

    local_seed_input = widgets.IntText(value=42, description='Seed:', style={'description_width': 'initial'})
    local_steps_input = widgets.IntSlider(
        value=4, min=1, max=50, step=1, description='Inference Steps:',
        style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'),
    )
    local_width_input = widgets.IntText(value=512, description='Width:', style={'description_width': 'initial'})
    local_height_input = widgets.IntText(value=512, description='Height:', style={'description_width': 'initial'})
    local_generate_image_button = widgets.Button(
        description='Generate Image', button_style='primary', layout=widgets.Layout(width='300px', height='50px'),
    )
    local_send_to_gallery_button = widgets.Button(
        description='Send to Gallery', button_style='info', layout=widgets.Layout(width='300px', height='50px'),
    )
    local_generation_output = widgets.Output()
    local_gallery_output = widgets.Output()

    def local_generate_from_embedding(b):
        """This function is a closure over the local widgets created above."""
        nonlocal last_generated_polaroid, last_generated_name # Declare them as nonlocal to modify outer scope variables
        with local_generation_output:
            local_generation_output.clear_output()
            local_gallery_output.clear_output() # Clear any previous gallery message

            name = local_gen_embedding_dropdown.value
            if not name:
                print("❌ No embedding selected! Save one in an earlier section first.")
                return
            entry = embedding_library[name]
            embedding, prompt = entry["embedding"], entry["prompt"]

            print(f"Generating image from '{name}'")
            print(f"  Prompt: '{prompt}'")
            print(f"  Shape: {embedding.shape}")
            print(f"  Seed: {local_seed_input.value}  Steps: {local_steps_input.value}  Size: {local_width_input.value}x{local_height_input.value}")

            embeds = torch.from_numpy(embedding.astype(np.float32)).to(device=device, dtype=torch.float16).unsqueeze(0)

            try:
                print(f"\n{'='*60}\nRunning FLUX.2 [klein] diffusion...\n{'='*60}\n")
                image = flux_pipe(
                    prompt_embeds=embeds,
                    num_inference_steps=local_steps_input.value,
                    guidance_scale=1.0,  # ignored anyway — klein is distilled, like FLUX.1-schnell
                    height=local_height_input.value,
                    width=local_width_input.value,
                    generator=torch.manual_seed(local_seed_input.value),
                ).images[0]

                safe_name = "".join(c if c.isalnum() else "_" for c in name)[:60]
                output_path = OUTPUT_IMAGES_DIR / f"{safe_name}.png"
                polaroid = make_polaroid(image, [name, "FLUX.2 [klein]-4B"])
                polaroid.save(output_path)

                last_generated_polaroid = polaroid # Store for gallery button
                last_generated_name = safe_name    # Store for gallery button

                print(f"✓ Saved to {output_path}")
                display(polaroid)
            except Exception as e:
                print(f"❌ Error generating image: {e}")
                import traceback
                traceback.print_exc()

    def local_send_to_gallery(b):
        nonlocal last_generated_polaroid, last_generated_name
        with local_gallery_output:
            local_gallery_output.clear_output()
            if last_generated_polaroid is None:
                print("❌ No image generated yet to send to gallery!")
                return

            gallery_dir = MODELS_DIR / "Gallery"
            gallery_dir.mkdir(parents=True, exist_ok=True)

            # Create a unique filename for the gallery copy
            from datetime import datetime
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            gallery_filename = f"{last_generated_name}_{timestamp}.png"
            gallery_path = gallery_dir / gallery_filename

            try:
                last_generated_polaroid.save(gallery_path)
                print(f"✓ Image saved to gallery: {gallery_path}")
            except Exception as e:
                print(f"❌ Error saving to gallery: {e}")
                import traceback
                traceback.print_exc()


    local_generate_image_button.on_click(local_generate_from_embedding)
    local_send_to_gallery_button.on_click(local_send_to_gallery)

    return widgets.VBox([
        widgets.HTML("<h2>Embedding Selection & Image Generation</h2>"),
        local_gen_embedding_dropdown,
        widgets.HTML("<br><h3>Generation Controls</h3>"),
        local_seed_input, local_steps_input,
        widgets.HTML("<br><b>Image Dimensions:</b>"),
        widgets.HBox([local_width_input, local_height_input]),
        widgets.HTML("<br>"),
        local_generate_image_button,
        widgets.HTML("<br><h3>Generated Image</h3>"),
        local_generation_output,
        local_send_to_gallery_button,
        local_gallery_output
    ])

# Create two independent VBox instances
vbox1 = create_image_generation_vbox()
vbox2 = create_image_generation_vbox()

# Display them side-by-side
display(widgets.HBox([vbox1, vbox2]))

# FLUX.2 [klein] Scaling Animation

In [ ]:
SEQUENCE_DIR.mkdir(parents=True, exist_ok=True)

# Set by generate_animation_frames() once a run finishes; read by the export cell below.
last_animation_dir = None
last_animation_num_frames = None

# --- Animation Type Selection ---
anim_type_dropdown = widgets.Dropdown(
    options=['Scale Single Embedding', 'Blend Two Embeddings'],
    value='Scale Single Embedding', description='Animation Type:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='auto')
)

# --- Widgets for Scale Single Embedding ---
anim_embedding_dropdown = widgets.Dropdown(
    options=list(embedding_library.keys()), description='Embedding (A):',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='600px'),
)
embedding_dropdowns.append(anim_embedding_dropdown)

anim_start_scale_input = widgets.FloatText(
    value=1.0, description='Start Scale:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'),
)
anim_end_scale_input = widgets.FloatText(
    value=2.0, description='End Scale:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'),
)

# --- Widgets for Blend Two Embeddings ---
anim_embedding_a_dropdown = widgets.Dropdown(
    options=list(embedding_library.keys()), description='Embedding (A):',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='600px'),
)
embedding_dropdowns.append(anim_embedding_a_dropdown)

anim_embedding_b_dropdown = widgets.Dropdown(
    options=list(embedding_library.keys()), description='Embedding (B):',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='600px'),
)
embedding_dropdowns.append(anim_embedding_b_dropdown)

anim_blend_start_alpha_input = widgets.FloatText(
    value=0.0, description='Start Alpha (for B):', style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'),
)
anim_blend_end_alpha_input = widgets.FloatText(
    value=1.0, description='End Alpha (for B):', style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'),
)

# --- General Animation Settings ---
num_frames_input = widgets.IntText(
    value=30, description='Number of frames:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'),
)
anim_seed_input = widgets.IntText(value=42, description='Seed:', style={'description_width': 'initial'})
anim_steps_input = widgets.IntSlider(
    value=4, min=1, max=50, step=1, description='Inference Steps:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'),
)
anim_width_input = widgets.IntText(value=512, description='Width:', style={'description_width': 'initial'})
anim_height_input = widgets.IntText(value=512, description='Height:', style={'description_width': 'initial'})
anim_generate_button = widgets.Button(
    description='Generate Animation Frames', button_style='primary',
    layout=widgets.Layout(width='300px', height='50px'),
)
anim_status_label = widgets.HTML(value='')
anim_generation_output = widgets.Output()

# Container for animation type specific parameters
anim_param_output_container = widgets.VBox()

def update_animation_parameters(animation_type):
    # Clear current children by setting to an empty tuple/list
    anim_param_output_container.children = ()
    if animation_type == 'Scale Single Embedding':
        anim_param_output_container.children = (
            anim_embedding_dropdown,
            widgets.HBox([anim_start_scale_input, anim_end_scale_input])
        )
    elif animation_type == 'Blend Two Embeddings':
        anim_param_output_container.children = (
            anim_embedding_a_dropdown,
            anim_embedding_b_dropdown,
            widgets.HBox([anim_blend_start_alpha_input, anim_blend_end_alpha_input])
        )

# Observe changes in anim_type_dropdown to update parameters
anim_type_dropdown.observe(lambda change: update_animation_parameters(change.new), names='value')

# Initial call to display default parameters
update_animation_parameters(anim_type_dropdown.value)

def generate_animation_frames(b):
    global last_animation_dir, last_animation_num_frames
    with anim_generation_output:
        anim_generation_output.clear_output()

        animation_type = anim_type_dropdown.value
        num_frames = num_frames_input.value
        if num_frames < 1:
            print("Number of frames must be at least 1.")
            return

        from datetime import datetime
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_dir = SEQUENCE_DIR / f"{animation_type.replace(' ', '_').lower()}_{timestamp}"
        output_dir.mkdir(parents=True, exist_ok=True)

        print(f"Generating {num_frames} frames for '{animation_type}'")
        print(f"  Output dir: {output_dir}")
        print(f"  Seed: {anim_seed_input.value}  Steps: {anim_steps_input.value}")
        print(f"  Dimensions: {anim_width_input.value}x{anim_height_input.value}")

        anim_generate_button.disabled = True
        anim_generate_button.description = 'Generating…'
        anim_status_label.value = '<b style="color:#b8860b;">⏳ Generating…</b>'

        frames_saved = 0
        try:
            for i in range(num_frames):
                progress = i / (num_frames - 1) if num_frames > 1 else 0.0
                current_embedding_np = None
                label_line = ""

                if animation_type == 'Scale Single Embedding':
                    name = anim_embedding_dropdown.value
                    if not name:
                        print("❌ No embedding selected for scaling!")
                        return
                    base_embedding = embedding_library[name]["embedding"]
                    base_tensor = torch.from_numpy(base_embedding.astype(np.float32)).to(device=device, dtype=torch.float16).unsqueeze(0)

                    start_scale = anim_start_scale_input.value
                    end_scale = anim_end_scale_input.value
                    current_scale = start_scale + (end_scale - start_scale) * progress
                    current_embedding_tensor = base_tensor * current_scale
                    current_embedding_np = current_embedding_tensor.cpu().numpy()[0]
                    label_line = f"{name} (Scale: {current_scale:.2f})"

                elif animation_type == 'Blend Two Embeddings':
                    name_a = anim_embedding_a_dropdown.value
                    name_b = anim_embedding_b_dropdown.value
                    if not name_a or not name_b:
                        print("❌ Both embeddings A and B must be selected for blending!")
                        return
                    emb_a_np = embedding_library[name_a]["embedding"]
                    emb_b_np = embedding_library[name_b]["embedding"]

                    emb_a_tensor = torch.from_numpy(emb_a_np.astype(np.float32)).to(device=device, dtype=torch.float16).unsqueeze(0)
                    emb_b_tensor = torch.from_numpy(emb_b_np.astype(np.float32)).to(device=device, dtype=torch.float16).unsqueeze(0)

                    start_alpha = anim_blend_start_alpha_input.value
                    end_alpha = anim_blend_end_alpha_input.value
                    current_alpha = start_alpha + (end_alpha - start_alpha) * progress

                    current_embedding_tensor = (emb_a_tensor * (1.0 - current_alpha)) + (emb_b_tensor * current_alpha)
                    current_embedding_np = current_embedding_tensor.cpu().numpy()[0]
                    label_line = f"Blend: {name_a} ({1.0-current_alpha:.2f}) + {name_b} ({current_alpha:.2f})"

                if current_embedding_np is None:
                    print(f"❌ Failed to prepare embedding for frame {i}")
                    continue

                try:
                    image = flux_pipe(
                        prompt_embeds=torch.from_numpy(current_embedding_np.astype(np.float32)).to(device=device, dtype=torch.float16).unsqueeze(0),
                        num_inference_steps=anim_steps_input.value,
                        guidance_scale=1.0,
                        height=anim_height_input.value,
                        width=anim_width_input.value,
                        generator=torch.manual_seed(anim_seed_input.value),
                    ).images[0]

                    polaroid = make_polaroid(image, [label_line, f"Frame {i+1}/{num_frames}", "FLUX.2 [klein]-4B"])
                    polaroid.save(output_dir / f"frame_{i:04d}.png")
                    frames_saved += 1
                    print(f"  Frame {i:04d}/{num_frames-1} saved")
                except Exception as e:
                    print(f"  Error on frame {i}: {e}")
                    import traceback
                    traceback.print_exc()
                    break

            if frames_saved == num_frames:
                last_animation_dir = output_dir
                last_animation_num_frames = frames_saved
                print(f"\n✅ Done! {frames_saved} frames saved to {output_dir}")
                anim_status_label.value = f'<b style="color:green;">✅ Done — {frames_saved} frames saved to {output_dir.name}. Export it below.</b>'
            else:
                print(f"\n⚠️ Stopped after {frames_saved}/{num_frames} frames — see the error above.")
                anim_status_label.value = f'<b style="color:#b00020;">⚠️ Stopped after {frames_saved}/{num_frames} frames.</b>'
        finally:
            anim_generate_button.disabled = False
            anim_generate_button.description = 'Generate Animation Frames'

anim_generate_button.on_click(generate_animation_frames)

display(widgets.VBox([
    widgets.HTML("<h2>Embedding Transformation Animation</h2>"),
    anim_type_dropdown,
    anim_param_output_container, # This container will display the type-specific widgets
    widgets.HTML("<br><h3>Generation Settings</h3>"),
    num_frames_input,
    anim_seed_input, anim_steps_input,
    widgets.HTML("<b>Image Dimensions:</b>"),
    widgets.HBox([anim_width_input, anim_height_input]),
    widgets.HTML("<br>"),
    anim_generate_button,
    anim_status_label,
    widgets.HTML("<br><h3>Output</h3>"),
    anim_generation_output,
]))


## Export the Animation

Turn the saved frames into a single animated **WebP**, preview it inline, and optionally copy it
into the shared gallery folder for others to see.


In [ ]:
# @title Export Animation: Save as WebP, Preview, and Send to Gallery
from IPython.display import Image as IPyImage

anim_export_fps_input = widgets.IntSlider(
    value=12, min=1, max=30, step=1, description='FPS:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='400px'),
)
anim_export_button = widgets.Button(description='Save & Preview WebP', button_style='success')
anim_export_gallery_button = widgets.Button(description='Send to Gallery', button_style='info')
anim_export_output = widgets.Output()

last_animation_webp_path = None

def export_animation_webp(b):
    global last_animation_webp_path
    with anim_export_output:
        anim_export_output.clear_output()

        if not last_animation_dir or not last_animation_dir.exists():
            print("❌ No finished animation yet — run 'Generate Animation Frames' above first.")
            return

        frame_paths = sorted(last_animation_dir.glob("frame_*.png"))
        if not frame_paths:
            print(f"❌ No frames found in {last_animation_dir}")
            return

        frames = [Image.open(p) for p in frame_paths]
        webp_path = last_animation_dir.with_suffix(".webp")
        duration_ms = round(1000 / anim_export_fps_input.value)
        frames[0].save(
            webp_path, format="WEBP", save_all=True, append_images=frames[1:],
            duration=duration_ms, loop=0, quality=90,
        )
        last_animation_webp_path = webp_path

        print(f"✓ Saved {len(frames)} frames as {webp_path.name} ({anim_export_fps_input.value} fps)")
        print(f"  Size: {webp_path.stat().st_size / 1024:.1f} KB")
        display(IPyImage(filename=str(webp_path)))

def send_animation_to_gallery(b):
    with anim_export_output:
        if not last_animation_webp_path:
            print("❌ Nothing to send yet — click 'Save & Preview WebP' first.")
            return
        gallery_dir = MODELS_DIR / "Gallery"
        gallery_dir.mkdir(parents=True, exist_ok=True)

        # The filename already carries a generation timestamp (from the frames' output dir),
        # so it's unique on its own — no need to tack on another one.
        gallery_path = gallery_dir / last_animation_webp_path.name
        shutil.copy(last_animation_webp_path, gallery_path)
        print(f"✓ Sent to gallery: {gallery_path}")

anim_export_button.on_click(export_animation_webp)
anim_export_gallery_button.on_click(send_animation_to_gallery)

display(widgets.VBox([
    anim_export_fps_input,
    widgets.HBox([anim_export_button, anim_export_gallery_button]),
    anim_export_output,
]))


---